# Stage 4 - adversarial attacks (FGSM / PGD, transfer to both models)

Wrap the CNN in ART, craft FGSM and PGD examples in the scaled space, and feed the SAME crafted frames to both the CNN and the Random Forest (a fair transfer test). Then cross-validate per-class robustness across the diversity gradient.

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Setup - retrain the CNN + RF (ART needs the live model for gradients)

In [ ]:
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn, build_random_forest
from adversec.experiments import attack as atk
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)
def mf1(y, p): return f1_score(y, p, average='macro', zero_division=0)

## Epsilon sweep - macro-F1 under FGSM and PGD (both models)
The same crafted frames hit both models. Watch where each collapses.

In [ ]:
wrapped = {}
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    Xtr, ytr, Xte, yte = a['X_train'], a['y_train'], a['X_test'].astype(np.float32), a['y_test']
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    cfg = config.load_dataset_config(name)
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    cnn = train_cnn(CNN1D(n_features=Xtr.shape[1], n_classes=len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw)
    rf = build_random_forest().fit(Xtr, ytr)
    clf = atk.wrap_cnn_for_art(cnn, n_features=Xtr.shape[1], n_classes=len(classes), device=DEVICE)
    wrapped[name] = (clf, rf, Xte, yte, classes)
    print(f'\n=== {name} ===')
    print(f"{'eps':>6}{'FGSM_CNN':>10}{'FGSM_RF':>9}{'PGD_CNN':>9}{'PGD_RF':>9}")
    cc, cr = mf1(yte, clf.predict(Xte).argmax(1)), mf1(yte, rf.predict(Xte))
    print(f"{'clean':>6}{cc:>10.3f}{cr:>9.3f}{cc:>9.3f}{cr:>9.3f}")
    for eps in config.FGSM_EPSILONS:
        Xf = atk.generate_fgsm(clf, Xte, eps); Xp = atk.generate_pgd(clf, Xte, eps)
        print(f'{eps:>6.2f}{mf1(yte, clf.predict(Xf).argmax(1)):>10.3f}{mf1(yte, rf.predict(Xf)):>9.3f}'
              f'{mf1(yte, clf.predict(Xp).argmax(1)):>9.3f}{mf1(yte, rf.predict(Xp)):>9.3f}')

## Cross-validated per-class robustness vs the diversity gradient
Slower (5-fold, retrains per fold). Shows that robustness does NOT follow signature count - the falsified hypothesis.

In [ ]:
from adversec.experiments.crossval import crossval_perclass_robustness
for name in DATASETS:
    strict = pd.read_csv(config.PROCESSED_DIR / f'{name}_strict.csv')
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    print(f'\n=== {name}: 5-fold per-class F1 under PGD ===')
    res = crossval_perclass_robustness(strict, FEATURES, classes, config.FGSM_EPSILONS,
                                       attack='pgd', n_splits=5, device=DEVICE)
    sig = strict[LABEL_COLUMN].value_counts()
    rows = []
    for i, c in enumerate(classes):
        row = {'class': c, 'signatures': int(sig.get(c, 0))}
        for e in ['clean'] + config.FGSM_EPSILONS:
            row[str(e)] = round(float(np.mean(res[e][i])), 3)
        rows.append(row)
    display(pd.DataFrame(rows).sort_values('signatures', ascending=False))